In [10]:
%pip install -q ultralytics==8.4.1 pycocotools==2.0.11 tqdm
!git clone https://github.com/mdmohsin212/WasteScope.git

Cloning into 'WasteScope'...
remote: Enumerating objects: 512, done.
remote: Counting objects: 100% (512/512), done.
remote: Compressing objects: 100% (282/282), done.
remote: Total 512 (delta 281), reused 421 (delta 196), pack-reused 0 (from 0)
Receiving objects: 100% (512/512), 28.85 MiB | 21.25 MiB/s, done.
Resolving deltas: 100% (281/281), done.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import hashlib
import importlib.util
import json
import math
import platform
import shutil
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from tqdm.auto import tqdm
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
BASE = Path("/content/drive/MyDrive/WasteScope")
RUN_NAME = "S2_yolo26n_seg_roi_960"
WORK = Path("/content/wastescope_finalize") / RUN_NAME

IMGSZ, ROI_BATCH = 960, 8
CONF, AP_CONF, MAX_DET = 0.20, 0.001, 100

TRAIN_RUN = BASE / "runs/segment" / RUN_NAME
REPORT = BASE / "reports/segmentation" / RUN_NAME / "test_best_960_fused_conf0.2"

DATA = WORK / "data"
WORK.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

In [5]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def environment():
    return {"python": platform.python_version(), "torch": torch.__version__,
            "ultralytics": version("ultralytics"), "pycocotools": version("pycocotools"),
            "numpy": np.__version__, "opencv": cv2.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

print("Reports:", REPORT)

Reports: /content/drive/MyDrive/WasteScope/reports/segmentation/S2_yolo26n_seg_roi_960/test_best_960_fused_conf0.2


In [ ]:
policy = json.loads((TRAIN_RUN / "roi_policy.json").read_text())

checkpoint_sources = {
    "detector": BASE / "runs/detect/D1_yolo26n_960/weights/best.pt",
    "full_image": BASE / "runs/segment/S1_yolo26n_seg_960/weights/best.pt",
    "roi": TRAIN_RUN / "weights/best.pt"
}

expected_hashes = {
    "detector": "cd6b987150bb2106321e59b61ac080a043d451994eb1e1cf99bd192a271109ea",
    "full_image": "1b8b41a691c0e51691565d4a50ffe7db50aaab144050b62b040548eae547a670",
    "roi": "c819c5b1e82228f33fd86e72365f8f841898acd88de2411f9e4541684256356f"
}

weights = {}
for branch, source in checkpoint_sources.items():
    target = WORK / f"{branch}.pt"
    shutil.copy2(source, target)
    weights[branch] = target

helper_path = WORK / "s2_monitor.py"
shutil.copy2(TRAIN_RUN / "s2_monitor.py", helper_path)
mask_pipeline = load_module("frozen_s2_masks", helper_path)

In [ ]:
EVALUATOR_COMMIT = "06e25e004fdf1b70d1a18c83511da71e835ae754"
evaluator_path = WORK / "metrics.py"

url = (f"https://raw.githubusercontent.com/mdmohsin212/WasteScope/"
       f"{EVALUATOR_COMMIT}/src/evaluation/metrics.py")

with urlopen(url, timeout=60) as response:
    evaluator_path.write_bytes(response.read())
if sha256(evaluator_path) != "76d0d88644f99c3ae7f0ceb0db6cf562ed0984ed2c3cd6cc2d11b07aad1e7c8b":
    raise ValueError("Evaluator source differs from the validated evaluator.")
canonical = load_module("frozen_segmentation_metrics", evaluator_path)


manifest_source = TRAIN_RUN / "video_split.json"
selection = {
    "system": "S2_fused", "run": RUN_NAME, "selected_on": "val", "result_stage": "final",
    "operating_confidence": CONF, "confidence_rule": ">", "imgsz": IMGSZ,
    "selection_reason": "Best canonical validation mask AP50-95; confidence 0.20 reduces false alarms with near-best validation F1.",
    "validation": {"mask_AP50": 0.6649369397204824, "mask_mAP50_95": 0.3650212107598986,
                   "confidence": 0.20, "TP": 471, "FP": 144, "FN": 390,
                   "precision": 471 / 615, "recall": 471 / 861, "F1": 942 / 1476,
                   "background_false_alarm_images": 84, "background_images": 401},
    "checkpoints": {key: {"path": str(path.relative_to(BASE)), "sha256": expected_hashes[key]}
                    for key, path in checkpoint_sources.items()},
    "roi_best_epoch": 1, "roi_policy": policy,
    "test_proposals": "D1 predictions only; no GT proposals, jitter or full-frame ROI fallback",
    "inference": {"AP_confidence_min": AP_CONF, "max_detections": MAX_DET,
                  "detector_max_det": 300, "half": True, "rect": False,
                  "retina_masks": True, "roi_batch": ROI_BATCH,
                  "fusion": "score-ordered mask-IoU NMS >= 0.5; S1 first on score ties"},
    "manifest_sha256": sha256(manifest_source),
    "evaluator_commit": EVALUATOR_COMMIT, "evaluator_source_sha256": sha256(evaluator_path),
    "fusion_helper_sha256": sha256(helper_path),
    "versions": {"ultralytics": "8.4.1", "pycocotools": "2.0.11"},
    "checkpoint_selection_limitation": "S0/S1 used native selection; S2 used canonical fused AP. Not a controlled architecture-only comparison.",
}

selection_id = hashlib.sha256(json.dumps(selection, sort_keys=True).encode()).hexdigest()
frozen_path = REPORT / "selected_system.json"
if frozen_path.is_file() and json.loads(frozen_path.read_text()) != selection:
    raise ValueError("This report already has a different frozen selection. Use a new report folder.")
write_json(frozen_path, selection)
for source in (helper_path, evaluator_path, manifest_source, TRAIN_RUN / "roi_policy.json"):
    shutil.copy2(source, REPORT / source.name)
write_json(REPORT / "evaluation_config.json", {"status": "partial", "selection_id": selection_id})
print("Frozen: S2 fused | confidence >", CONF, "| checkpoint epoch 1")